In [5]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec  #grid pels subplots
from matplotlib import colors         #colors
from matplotlib.colors import LogNorm   #normalitza a 0-1 en escala logarítmica 
from matplotlib import patches     #figures
from mpl_toolkits.mplot3d import Axes3D   #eixos en 3D
from mpl_toolkits.axes_grid1 import make_axes_locatable  #per canviar la posició dels eixos
from matplotlib.ticker import NullFormatter   #marques(tics) sense etiquetes en els eixos

from astroML.density_estimation import XDGMM
from astroML.plotting.tools import draw_ellipse
from astroML.plotting import scatter_contour
from astroML.crossmatch import crossmatch
from astroML.datasets import fetch_sdss_S82standards, fetch_imaging_sample
from astroML.stats import sigmaG
from astroML.utils.decorators import pickle_results
import os
import pickle

import astropy.table     #paquet per manejar taules de dades
from astropy.table import Table, Column, MaskedColumn   #importa taules, columnes i columnes que emmascaren dades invàlides
from astropy.visualization import astropy_mpl_style  #visualització 
from scipy.stats import gaussian_kde  #representation of a kernel-density estimate using Gaussian kernels.
import seaborn as sns  #llibreria per fer gràfics estadístics
import os.path   #per implementar diferents funcions amb pathnames ("dreceres")

from time import time   #mòdul de funcions de time access 
from sklearn import manifold, datasets #manifold: algoritme de dimensionality reduction 
                                       #sklearn (sci-kit learn): llibreria de Python per machine learning

import obtain_data   #per importar dades d'altres fitxers


import importlib   #package per importar coses a python
importlib.reload(obtain_data)

galah_rc = obtain_data.galah_dr3_rc()  

galah_rc.get_ndimspace(feh=True, norm="stdev")

X = galah_rc.X            #(10941, 24)
Xerr = galah_rc.Xerr1         #(10941, 24)
Xcov = np.zeros(Xerr.shape + Xerr.shape[-1:])
Xcov[:, range(Xerr.shape[1]), range(Xerr.shape[1])] = Xerr ** 2    #(10941, 24, 24)

In [36]:
def show_abundances(n_components):
    N = 0
    fe_h_mean = 0
    Z = [26,8,11,12,13,14,19,20,21,22,23,24,25,27,28,29,30,39,40,56,57,58,60,63]
    Z_labels = ["Fe","O","Na","Mg","Al","Si","K","Ca","Sc","Ti","V","Cr","Mn","Co","Ni","Cu","Zn","Y","Zr","Ba","La","Ce","Nd","Eu"]
    abundance_titles   = ['[Fe/H]' ,  '[O/Fe]',   '[Na/Fe]', '[Mg/Fe]',
                          '[Al/Fe]',  '[Si/Fe]',  '[K/Fe]',  '[Ca/Fe]',
                          '[Sc/Fe]',  '[Ti/Fe]',  '[V/Fe]' , '[Cr/Fe]',
                          '[Mn/Fe]',  '[Co/Fe]',  '[Ni/Fe]', '[Cu/Fe]',
                          '[Zn/Fe]',  '[Y/Fe]',  ' [Zr/Fe]', '[Ba/Fe]', 
                          '[La/Fe]',  '[Ce/Fe]',  '[Nd/Fe]', '[Eu/Fe]']
    ck = ["Black", "Red", "Gold", "Green", "Blue",
          "Orange", "Cyan", "Lime", "Magenta", "Yellow",
          "Indianred", "Hotpink", "Peru", "Cornflowerblue", "Olivedrab",
          "Grey", "Turquoise", "Lightpink", "Navy", "Khaki",
          "Darkgreen", "Crimson", "Deepskyblue", "Sandybrown", "Limegreen",
          "Deeppink", "Dodgerblue", "Rebeccapurple", "Teal", "Magenta"]
    
    print('Component;   colour;   % in sample; Mean [Fe/H];  [Fe/H] dispersion;    max[X/Fe] ;    min[X/Fe];   Most discr. [X/Fe] (w Sun)')

    for ii in range (n_components):  # pels 5 clusters
        n = collections.Counter(maxprob_per_star)[ii] # n stars per cluster
        
        X_mean = np.sum(X[maxprob_per_star==ii], axis=0) / n  # mean abundances per cluster
        fe_h = X_mean[0]  # metalicitat mitjana del cluster ii
        Xerr_mean = np.sum(Xerr[maxprob_per_star==ii], axis=0) / n  # mean error abundances per cluster
        e_fe_h = Xerr_mean[0]
        
        x_max = abundance_titles[np.argmax(X_mean)]
        x_min = abundance_titles[np.argmin(X_mean)]
        if np.abs(np.max(X_mean)) < np.abs(np.min(X_mean)):
            x_discrep = x_min
        else:
            x_discrep = x_max

        N += n
        fe_h_mean += fe_h / n_components
        
        print(ii,';', ck[ii],'\t \t ', "{:.1f}".format(n*100/10941),'%', ' \t ', "{:.3f}".format(fe_h),' \t ', "{:.3f}".format(e_fe_h),' \t ',x_max,' \t ',x_min,' \t ', x_discrep   )    
    
    print('\n')
    print('\n')

In [37]:
import collections
for n_components in [5,25]:

    @pickle_results('XD_'+str(n_components)+'clusters.pkl')
    def compute_XD(n_components, rseed=0, max_iter=100, verbose=True):
        np.random.seed(rseed)
        clf = XDGMM(n_components, max_iter=max_iter, tol=1E-5, verbose=verbose)
        clf.fit(X, Xcov) 
        return clf
    # Fit the model on training set
    clf = compute_XD(n_components)

    # Probabilities for each star to belong to each group
    logprob = (clf.logprob_a(X, Xcov))
    prob_per_star = np.exp(logprob) / np.sum(np.exp(logprob), axis=1)[:, np.newaxis]  #(10941 estrelles, 25 probs)
    maxprob_per_star = np.argmax(prob_per_star, axis=1)   #suma en horitzontal

    show_abundances(n_components)

@pickle_results: using precomputed results from 'XD_5clusters.pkl'
Component;   colour;   % in sample; Mean [Fe/H];  [Fe/H] dispersion;    max[X/Fe] ;    min[X/Fe];   Most discr [X/Fe] (w Sun)
0 ; Black 	 	  11.6 %  	  -0.024  	  0.060  	  [V/Fe]  	  [Ce/Fe]  	  [V/Fe]
1 ; Red 	 	  3.6 %  	  -0.309  	  0.066  	  [Ba/Fe]  	  [Fe/H]  	  [Ba/Fe]
2 ; Gold 	 	  37.1 %  	  -0.037  	  0.061  	  [V/Fe]  	  [Y/Fe]  	  [V/Fe]
3 ; Green 	 	  21.1 %  	  -0.308  	  0.064  	  [O/Fe]  	  [Fe/H]  	  [O/Fe]
4 ; Blue 	 	  26.6 %  	  -0.213  	  0.060  	  [Ba/Fe]  	  [Fe/H]  	  [Ba/Fe]




@pickle_results: using precomputed results from 'XD_25clusters.pkl'
Component;   colour;   % in sample; Mean [Fe/H];  [Fe/H] dispersion;    max[X/Fe] ;    min[X/Fe];   Most discr [X/Fe] (w Sun)
0 ; Black 	 	  2.2 %  	  -0.233  	  0.060  	  [V/Fe]  	  [Fe/H]  	  [V/Fe]
1 ; Red 	 	  8.6 %  	  -0.105  	  0.062  	  [Ba/Fe]  	  [Y/Fe]  	  [Ba/Fe]
2 ; Gold 	 	  5.8 %  	  -0.445  	  0.064  	  [O/Fe]  	  [Fe/H]  	  [Fe/H]
3 ; G